# 🚀 SPARK TUTORIAL 01: INTRODUCCIÓN BÁSICA

## 🎯 **OBJETIVO**
Aprender los conceptos fundamentales de Apache Spark

## 📋 **CONTENIDO**
- ¿Qué es Apache Spark?
- Configuración de SparkSession
- Conceptos básicos: RDD, DataFrames, Datasets
- Primeras operaciones
- Lazy Evaluation
- Trabajo con archivos

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [39]:
# 🔄 CELDA DE REINICIO - Ejecutar si hay errores de SparkContext
# Esta celda cierra cualquier sesión anterior y crea una nueva

try:
    if 'spark' in globals():
        print("🔄 Cerrando sesión anterior de Spark...")
        spark.stop()
        print("✅ Sesión anterior cerrada")
except:
    print("ℹ️ No había sesión anterior")

# Limpiar variables
if 'spark' in globals():
    del spark

print("🚀 Listo para crear nueva sesión de Spark")


🔄 Cerrando sesión anterior de Spark...
✅ Sesión anterior cerrada
🚀 Listo para crear nueva sesión de Spark


In [21]:
# Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum
import os

print("📚 Librerías importadas correctamente")


📚 Librerías importadas correctamente


## 🔧 **PASO 1: CREAR SPARKSESSION**

SparkSession es el punto de entrada principal para usar Spark SQL y DataFrames.


In [25]:
# Crear SparkSession para el curso
import socket
import os

# Detectar si estamos dentro de un contenedor Docker
def get_spark_master():
    try:
        # Intentar conectar al master de Docker
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname or 'jupyterlab' in hostname:
            return "spark://master:7077"  # Desde dentro del contenedor
        else:
            return "spark://localhost:7077"  # Desde fuera del contenedor
    except:
        return "local[*]"  # Fallback a modo local

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("EducacionIT-Spark-Basics") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "2g")\
    .config("spark.executor.cores", "1")\
    .config("spark.executor.instances", "1")\
    .config("spark.driver.memory", "1g") \
    .config("spark.driver.cores", "1") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada exitosamente")


🔧 Conectando a: spark://master:7077
✅ SparkSession creada exitosamente


In [26]:
spark

## 📊 **PASO 2: INFORMACIÓN DEL CLUSTER**

Vamos a ver información sobre nuestro cluster Spark.


In [27]:
# Información de la aplicación
print("🔧 Nombre de la aplicación:", spark.sparkContext.appName)
print("🆔 ID de la aplicación:", spark.sparkContext.applicationId)
print("🌐 URL de la UI:", spark.sparkContext.uiWebUrl)
print("📦 Versión de Spark:", spark.version)
print("🐍 Versión de Python:", spark.sparkContext.pythonVer)


🔧 Nombre de la aplicación: EducacionIT-Spark-Basics
🆔 ID de la aplicación: app-20250927045450-0004
🌐 URL de la UI: http://jupyterlab:4040
📦 Versión de Spark: 3.5.3
🐍 Versión de Python: 3.10


In [28]:
# Información del cluster (versión corregida)
print("🔍 Información del SparkContext:")
print(f"  Versión de Spark: {spark.version}")
print(f"  App Name: {spark.sparkContext.appName}")
print(f"  App ID: {spark.sparkContext.applicationId}")
print(f"  Master URL: {spark.sparkContext.master}")
print(f"  Número de cores disponibles: {spark.sparkContext.defaultParallelism}")

# Información de ejecutores (método compatible)
try:
    status = spark.sparkContext.statusTracker()
    executors = status.getExecutorInfos()
    print(f"\n👥 Número de ejecutores: {len(executors)}")
    print("\n💾 Información de ejecutores:")
    
    for i, executor in enumerate(executors, 1):
        print(f"  {i}. ID: {executor.executorId}")
        print(f"     Host: {executor.executorHost}")
        print(f"     Cores: {executor.totalCores}")
        print(f"     Memoria máxima: {executor.maxMemory}")
        print(f"     Estado: {executor.isActive}")
        print()
        
except AttributeError:
    print("\n👥 Información de ejecutores (API moderna):")
    print(f"  Número de cores: {spark.sparkContext.defaultParallelism}")
    print(f"  Master: {spark.sparkContext.master}")


🔍 Información del SparkContext:
  Versión de Spark: 3.5.3
  App Name: EducacionIT-Spark-Basics
  App ID: app-20250927045450-0004
  Master URL: spark://master:7077
  Número de cores disponibles: 2

👥 Información de ejecutores (API moderna):
  Número de cores: 2
  Master: spark://master:7077


## 📚 **PASO 3: TRABAJANDO CON RDDS**

Los RDDs (Resilient Distributed Datasets) son la abstracción fundamental de Spark.


In [29]:
# Crear RDD desde una lista
numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd_numeros = spark.sparkContext.parallelize(numeros)

print(f"🔢 RDD creado con {rdd_numeros.count()} elementos")
print(f"➕ Suma total: {rdd_numeros.sum()}")
print(f"📊 Promedio: {rdd_numeros.mean():.2f}")
print(f"📈 Máximo: {rdd_numeros.max()}")
print(f"📉 Mínimo: {rdd_numeros.min()}")


🔢 RDD creado con 10 elementos
➕ Suma total: 55
📊 Promedio: 5.50
📈 Máximo: 10
📉 Mínimo: 1


In [30]:
# Transformaciones con RDD
rdd_pares = rdd_numeros.filter(lambda x: x % 2 == 0)
print(f"🔢 Números pares: {rdd_pares.collect()}")

rdd_cuadrados = rdd_numeros.map(lambda x: x ** 2)
print(f"📐 Cuadrados: {rdd_cuadrados.collect()}")


🔢 Números pares: [2, 4, 6, 8, 10]
📐 Cuadrados: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


## 📊 **PASO 4: TRABAJANDO CON DATAFRAMES**

Los DataFrames son más fáciles de usar que los RDDs y están optimizados.


In [31]:
# Crear DataFrame desde datos de ejemplo
datos_empleados = [
    ("Juan", "IT", 50000, 25),
    ("María", "Marketing", 45000, 30),
    ("Carlos", "IT", 55000, 28),
    ("Ana", "HR", 40000, 35),
    ("Luis", "IT", 60000, 32),
    ("Laura", "Marketing", 48000, 27)
]

columnas = ["nombre", "departamento", "salario", "edad"]
df_empleados = spark.createDataFrame(datos_empleados, columnas)

print("👥 DataFrame de empleados creado:")
df_empleados.show()


👥 DataFrame de empleados creado:


+------+------------+-------+----+
|nombre|departamento|salario|edad|
+------+------------+-------+----+
|  Juan|          IT|  50000|  25|
| María|   Marketing|  45000|  30|
|Carlos|          IT|  55000|  28|
|   Ana|          HR|  40000|  35|
|  Luis|          IT|  60000|  32|
| Laura|   Marketing|  48000|  27|
+------+------------+-------+----+



In [32]:
# Ver schema del DataFrame
print("📋 Schema del DataFrame:")
df_empleados.printSchema()


📋 Schema del DataFrame:
root
 |-- nombre: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)
 |-- edad: long (nullable = true)



In [33]:
# Operaciones básicas con DataFrames
print("🔍 EMPLEADOS DEL DEPARTAMENTO IT:")
df_empleados.filter(col("departamento") == "IT").show()

print("\n💰 EMPLEADOS CON SALARIO > 50000:")
df_empleados.filter(col("salario") > 50000).show()


🔍 EMPLEADOS DEL DEPARTAMENTO IT:


+------+------------+-------+----+
|nombre|departamento|salario|edad|
+------+------------+-------+----+
|  Juan|          IT|  50000|  25|
|Carlos|          IT|  55000|  28|
|  Luis|          IT|  60000|  32|
+------+------------+-------+----+


💰 EMPLEADOS CON SALARIO > 50000:
+------+------------+-------+----+
|nombre|departamento|salario|edad|
+------+------------+-------+----+
|Carlos|          IT|  55000|  28|
|  Luis|          IT|  60000|  32|
+------+------------+-------+----+



In [38]:
# Importar librerías avanzadas
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, coalesce, regexp_replace, split, explode,
    count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    row_number, rank, dense_rank, lag, lead, first, last,
    date_format, year, month, dayofmonth, datediff, current_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("📚 Librerías avanzadas importadas correctamente")
# DataFrame de empleados con más información
empleados_data = [
    (1, "Juan Pérez", "IT", 50000, "2020-01-15", "Madrid"),
    (2, "María García", "Marketing", 45000, "2019-03-20", "Barcelona"),
    (3, "Carlos López", "IT", 55000, "2021-06-10", "Madrid"),
    (4, "Ana Martínez", "HR", 40000, "2018-11-05", "Valencia"),
    (5, "Luis Rodríguez", "IT", 60000, "2017-09-12", "Sevilla"),
    (6, "Laura Sánchez", "Marketing", 48000, "2022-02-28", "Barcelona"),
    (7, "Pedro González", "Sales", 42000, "2020-07-15", "Madrid"),
    (8, "Carmen Ruiz", "IT", 52000, "2021-04-03", "Bilbao"),
    (9, "Miguel Torres", "HR", 38000, "2019-12-10", "Valencia"),
    (10, "Isabel Díaz", "Sales", 46000, "2020-08-22", "Sevilla")
]

empleados_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("salario", IntegerType(), True),
    StructField("fecha_ingreso", StringType(), True),
    StructField("ciudad", StringType(), True)
])

df_empleados = spark.createDataFrame(empleados_data, empleados_schema)
df_empleados = df_empleados.withColumn("fecha_ingreso", col("fecha_ingreso").cast(DateType()))

print("👥 DataFrame empleados:")
df_empleados.show()


📚 Librerías avanzadas importadas correctamente
👥 DataFrame empleados:
+---+--------------+------------+-------+-------------+---------+
| id|        nombre|departamento|salario|fecha_ingreso|   ciudad|
+---+--------------+------------+-------+-------------+---------+
|  1|    Juan Pérez|          IT|  50000|   2020-01-15|   Madrid|
|  2|  María García|   Marketing|  45000|   2019-03-20|Barcelona|
|  3|  Carlos López|          IT|  55000|   2021-06-10|   Madrid|
|  4|  Ana Martínez|          HR|  40000|   2018-11-05| Valencia|
|  5|Luis Rodríguez|          IT|  60000|   2017-09-12|  Sevilla|
|  6| Laura Sánchez|   Marketing|  48000|   2022-02-28|Barcelona|
|  7|Pedro González|       Sales|  42000|   2020-07-15|   Madrid|
|  8|   Carmen Ruiz|          IT|  52000|   2021-04-03|   Bilbao|
|  9| Miguel Torres|          HR|  38000|   2019-12-10| Valencia|
| 10|   Isabel Díaz|       Sales|  46000|   2020-08-22|  Sevilla|
+---+--------------+------------+-------+-------------+---------+



In [34]:
# Agregaciones
print("📊 ESTADÍSTICAS POR DEPARTAMENTO:")
df_empleados.groupBy("departamento").agg(
    count("nombre").alias("total_empleados"),
    spark_sum("salario").alias("salario_total"),
    spark_sum("edad").alias("edad_total")
).show()


📊 ESTADÍSTICAS POR DEPARTAMENTO:


[Stage 22:======================================>                 (51 + 2) / 75]

+------------+---------------+-------------+----------+
|departamento|total_empleados|salario_total|edad_total|
+------------+---------------+-------------+----------+
|          HR|              1|        40000|        35|
|   Marketing|              2|        93000|        57|
|          IT|              3|       165000|        85|
+------------+---------------+-------------+----------+



## ⚡ **PASO 5: LAZY EVALUATION**

Spark usa evaluación perezosa (lazy evaluation) para optimizar el procesamiento.


In [35]:
# Crear DataFrame grande para demostrar lazy evaluation
numeros = list(range(1, 1001))
df_numeros = spark.createDataFrame([(x,) for x in numeros], ["numero"])

print("🔢 DataFrame con 1000 números creado")

# Transformaciones (lazy)
df_transformado = df_numeros \
    .filter(col("numero") % 2 == 0) \
    .filter(col("numero") > 100) \
    .filter(col("numero") < 500) \
    .select((col("numero") * 2).alias("doble"))

print("🔄 Transformaciones aplicadas (sin ejecutar)")
print("⚡ Las transformaciones son 'lazy' - no se ejecutan hasta ahora")


🔢 DataFrame con 1000 números creado
🔄 Transformaciones aplicadas (sin ejecutar)
⚡ Las transformaciones son 'lazy' - no se ejecutan hasta ahora


In [36]:
# Acción (triggers execution)
print("🚀 Ejecutando acción - esto dispara el procesamiento...")
import time
start_time = time.time()

resultado = df_transformado.collect()

end_time = time.time()
print(f"✅ Resultado: {len(resultado)} elementos")
print(f"📊 Primeros 10: {[r.doble for r in resultado[:10]]}")
print(f"⏱️ Tiempo de ejecución: {end_time - start_time:.3f} segundos")


🚀 Ejecutando acción - esto dispara el procesamiento...
✅ Resultado: 199 elementos
📊 Primeros 10: [204, 208, 212, 216, 220, 224, 228, 232, 236, 240]
⏱️ Tiempo de ejecución: 0.559 segundos


## 🎯 **RESUMEN DEL TUTORIAL**

¡Felicitaciones! Has completado el tutorial básico de Spark.

### **📚 Conceptos aprendidos:**
- ✅ **SparkSession**: Punto de entrada principal
- ✅ **RDDs**: Estructura de datos distribuida básica
- ✅ **DataFrames**: Estructura optimizada para análisis
- ✅ **Transformaciones**: Operaciones que crean nuevos DataFrames
- ✅ **Acciones**: Operaciones que ejecutan el procesamiento
- ✅ **Lazy Evaluation**: Optimización automática de Spark

### **🚀 Próximos pasos:**
1. **Tutorial 02**: DataFrames Avanzados
2. **Tutorial 03**: Spark SQL
3. **Experimentar** con tus propios datos

---

**🎉 ¡Has dominado los fundamentos de Apache Spark!**


In [ ]:
# Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
